
# Olist Geolocation Dataset – Exploration & Cleaning

## 1. Dataset Overview

**Dataset:** olist_geolocation_dataset.csv

**Objective:**
Explore geographical information associated with Brazilian
ZIP-code prefixes, including coordinates, cities and states.

**Columns:**
1. geolocation_zip_code_prefix
2. geolocation_lat
3. geolocation_lng
4. geolocation_city
5. geolocation_state

**Business Relevance:**
- Understand geographical coverage across Brazil.
- Identify regional location concentration.
- Support customer and seller location analysis.
- Provide geographical information for supply chain analysis.

**Workflow:**
1. Data inspection
2. Data quality assessment
3. Exploratory data analysis
4. Data cleaning
5. Final validation
6. Export cleaned dataset


# Step 1 : Initial data inspection

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
# ZIP codes must remain strings to preserve leading zeros.
df_geo = pd.read_csv(
    "../data/raw/olist_geolocation_dataset.csv",
    dtype={
        "geolocation_zip_code_prefix": "string",
        "geolocation_city": "string",
        "geolocation_state": "string"
    }
)

In [3]:
print("Dataset Shape:", df_geo.shape)

display(df_geo.head())
display(df_geo.tail())

print("\nColumn Names:")
print(df_geo.columns.tolist())

print("\nData Types:")
print(df_geo.dtypes)

print("\nDataset Information:")
df_geo.info()

print("\nStatistical Summary:")
display(df_geo.describe(include="all").T)

Dataset Shape: (1000163, 5)


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01037,-23.545621,-46.639292,sao paulo,SP
1,01046,-23.546081,-46.644820,sao paulo,SP
2,01046,-23.546129,-46.642951,sao paulo,SP
3,01041,-23.544392,-46.639499,sao paulo,SP
4,01035,-23.541578,-46.641607,sao paulo,SP


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1000158,99950,-28.068639,-52.010705,tapejara,RS
1000159,99900,-27.877125,-52.224882,getulio vargas,RS
1000160,99950,-28.071855,-52.014716,tapejara,RS
1000161,99980,-28.388932,-51.846871,david canabarro,RS
1000162,99950,-28.070104,-52.018658,tapejara,RS



Column Names:
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Data Types:
geolocation_zip_code_prefix    string[python]
geolocation_lat                       float64
geolocation_lng                       float64
geolocation_city               string[python]
geolocation_state              string[python]
dtype: object

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  string 
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  string 
 4   geolocation_state            1000163 non-null  string 
dtypes: float64(2), string(3)
memory usage: 38.2 MB

Statistica

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
geolocation_zip_code_prefix,1000163,19015,24220,1146,NaN,NaN,NaN,NaN,NaN,NaN,NaN
geolocation_lat,1000163.0,NaN,NaN,NaN,-21.176153,5.715866,-36.605374,-23.603546,-22.919377,-19.97962,45.065933
geolocation_lng,1000163.0,NaN,NaN,NaN,-46.390541,4.269748,-101.466766,-48.573172,-46.637879,-43.767709,121.105394
geolocation_city,1000163,8011,sao paulo,135800,NaN,NaN,NaN,NaN,NaN,NaN,NaN
geolocation_state,1000163,27,SP,404268,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Initial Data Inspection – Findings

- Total records: 1,000,163
- Total columns: 5
- Memory usage: 38.2 MB
- Unique ZIP prefixes: 19,015
- Unique cities: 8,011
- Unique states: 27
- Missing values: 0

### Initial Observations
1. ZIP codes are correctly stored as strings.
2. Latitude and longitude are stored as float values.
3. São Paulo is the most frequently occurring city.
4. SP has the highest record count.
5. Some geographical coordinates require further investigation.

### Next Step
Perform data quality assessment and investigate
duplicate records and coordinate anomalies.

# Step 3 : Data quality assessment 

In [4]:
print("Missing Values:")
display(df_geo.isnull().sum())

print("\nExact Duplicate Rows:",
      df_geo.duplicated().sum())

print("\nDuplicate ZIP Prefixes:",
      df_geo["geolocation_zip_code_prefix"].duplicated().sum())

Missing Values:


geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64


Exact Duplicate Rows: 261831

Duplicate ZIP Prefixes: 981148


In [5]:
print("ZIP Length Distribution:")
display(
    df_geo["geolocation_zip_code_prefix"]
    .str.len()
    .value_counts()
    .sort_index()
)

print("Non-numeric ZIP codes:",
      (~df_geo["geolocation_zip_code_prefix"]
       .str.fullmatch(r"\d+")).sum())

ZIP Length Distribution:


geolocation_zip_code_prefix
5    1000163
Name: count, dtype: Int64

Non-numeric ZIP codes: 0


In [6]:
print("Latitude Range:")
print(df_geo["geolocation_lat"].min(),
      df_geo["geolocation_lat"]ax().m)

print("\nLongitude Range:")
print(df_geo["geolocation_lng"].min(),
      df_geo["geolocation_lng"].max())

Latitude Range:
-36.6053744107061 45.06593318269697

Longitude Range:
-101.46676644931476 121.10539381057764


In [7]:
outside_brazil = df_geo[
    ~df_geo["geolocation_lat"].between(-34, 6) |
    ~df_geo["geolocation_lng"].between(-74, -34)
]

print("Potentially Outside Brazil:", len(outside_brazil))

display(outside_brazil.head(10))

Potentially Outside Brazil: 42


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
513631,28165,41.614052,-8.411675,vila nova de campos,RJ
513643,28155,-34.586422,-58.732101,santa maria,RJ
513754,28155,42.439286,13.820214,santa maria,RJ
514429,28333,38.381672,-6.328200,raposo,RJ
516682,28595,43.684961,-7.411080,portela,RJ
538512,29654,29.409252,-98.484121,santo antônio do canaã,ES
538557,29654,21.657547,-101.466766,santo antonio do canaa,ES
585242,35179,25.995203,-98.078544,santana do paraíso,MG
585260,35179,25.995245,-98.078533,santana do paraiso,MG


In [8]:
invalid_lat = df_geo[
    ~df_geo["geolocation_lat"].between(-34, 6)
]

invalid_lng = df_geo[
    ~df_geo["geolocation_lng"].between(-74, -34)
]

print("Invalid latitude records:", len(invalid_lat))
print("Invalid longitude records:", len(invalid_lng))

Invalid latitude records: 31
Invalid longitude records: 37


In [9]:
outside_brazil = df_geo[
    ~df_geo["geolocation_lat"].between(-34, 6) |
    ~df_geo["geolocation_lng"].between(-74, -34)
]

display(outside_brazil.sort_values(
    ["geolocation_state", "geolocation_zip_code_prefix"]
))

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
727755,57319,45.065933,9.341528,pau d'arco,AL
695375,45936,-34.622400,-58.901888,itabata,BA
695377,45936,38.323939,-6.775035,itabatan,BA
697048,46560,38.991963,-4.947823,ibiajara,BA
698466,47310,38.268205,-7.803886,santana do sobrado,BA
538512,29654,29.409252,-98.484121,santo antônio do canaã,ES
538557,29654,21.657547,-101.466766,santo antonio do canaa,ES
585242,35179,25.995203,-98.078544,santana do paraíso,MG
585260,35179,25.995245,-98.078533,santana do paraiso,MG
822529,78131,38.816816,-9.394625,varzea grande,MT


In [10]:
display(
    outside_brazil.groupby("geolocation_state")
    .size()
    .reset_index(name="suspicious_records")
    .sort_values("suspicious_records", ascending=False)
)

,geolocation_state,suspicious_records
7,PE,11
5,PA,7
9,RJ,5
1,BA,4
10,RS,4
8,PR,3
2,ES,2
3,MG,2
0,AL,1
4,MT,1


In [11]:
# Display all suspicious records
pd.set_option("display.max_rows", None)

display(
    outside_brazil[
        [
            "geolocation_zip_code_prefix",
            "geolocation_lat",
            "geolocation_lng",
            "geolocation_city",
            "geolocation_state"
        ]
    ]
)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
513631,28165,41.614052,-8.411675,vila nova de campos,RJ
513643,28155,-34.586422,-58.732101,santa maria,RJ
513754,28155,42.439286,13.820214,santa maria,RJ
514429,28333,38.381672,-6.328200,raposo,RJ
516682,28595,43.684961,-7.411080,portela,RJ
538512,29654,29.409252,-98.484121,santo antônio do canaã,ES
538557,29654,21.657547,-101.466766,santo antonio do canaa,ES
585242,35179,25.995203,-98.078544,santana do paraíso,MG
585260,35179,25.995245,-98.078533,santana do paraiso,MG


In [12]:
outside_brazil.groupby("geolocation_state").agg(
    suspicious_records=("geolocation_state", "size"),
    unique_zip_codes=("geolocation_zip_code_prefix", "nunique"),
    unique_cities=("geolocation_city", "nunique")
).sort_values("suspicious_records", ascending=False)

,suspicious_records,unique_zip_codes,unique_cities
geolocation_state,,,
PE,11,1,1
PA,7,3,3
RJ,5,4,4
BA,4,3,4
RS,4,2,2
PR,3,2,2
ES,2,1,2
MG,2,1,2
AL,1,1,1


In [13]:
zip_summary = (
    df_geo.groupby("geolocation_zip_code_prefix")
    .agg(
        city_count=("geolocation_city", "nunique"),
        state_count=("geolocation_state", "nunique")
    )
)

# ZIP codes associated with multiple cities or states
zip_conflicts = zip_summary[
    (zip_summary["city_count"] > 1) |
    (zip_summary["state_count"] > 1)
]

print("Unique ZIP prefixes:", df_geo["geolocation_zip_code_prefix"].nunique())
print("ZIP conflicts:", len(zip_conflicts))

display(zip_conflicts.head(20))

Unique ZIP prefixes: 19015
ZIP conflicts: 8559


,city_count,state_count
geolocation_zip_code_prefix,,
01001,2,1
01002,2,1
01003,2,1
01004,2,1
01005,2,1
01006,2,1
01007,2,1
01008,2,1
01009,2,1


In [14]:
state_conflicts = zip_summary[
    zip_summary["state_count"] > 1
]

city_only_conflicts = zip_summary[
    (zip_summary["city_count"] > 1) &
    (zip_summary["state_count"] == 1)
]

print("ZIPs linked to multiple states:", len(state_conflicts))
print("ZIPs linked to multiple cities within one state:", len(city_only_conflicts))

ZIPs linked to multiple states: 8
ZIPs linked to multiple cities within one state: 8551


In [15]:
conflict_zips = state_conflicts.index

display(
    df_geo[
        df_geo["geolocation_zip_code_prefix"].isin(conflict_zips)
    ][
        [
            "geolocation_zip_code_prefix",
            "geolocation_city",
            "geolocation_state"
        ]
    ].drop_duplicates().sort_values("geolocation_zip_code_prefix")
)

,geolocation_zip_code_prefix,geolocation_city,geolocation_state
21728,02116,sao paulo,SP
22193,02116,são paulo,SP
22261,02116,são paulo,RN
71032,04011,sao paulo,SP
71247,04011,são paulo,SP
72852,04011,sao paulo,AC
430342,21550,rio de janeiro,RJ
430999,21550,rio de janeiro,AC
460406,23056,rio de janeiro,AC
459234,23056,rio de janeiro,RJ


In [16]:
df_geo_clean = df_geo.drop_duplicates().copy()

print("Original records:", len(df_geo))
print("Cleaned records:", len(df_geo_clean))
print("Removed exact duplicates:", len(df_geo) - len(df_geo_clean))
print("Remaining duplicates:", df_geo_clean.duplicated().sum())
print("Missing values:", df_geo_clean.isnull().sum().sum())

Original records: 1000163
Cleaned records: 738332
Removed exact duplicates: 261831
Remaining duplicates: 0
Missing values: 0


In [17]:
df_geo_clean.to_csv(
    "../data/processed/geolocation_clean.csv",
    index=False
)

print("Geolocation dataset exported successfully!")

Geolocation dataset exported successfully!


## Geolocation Dataset – Cleaning Summary

- Original records: 1,000,163
- Exact duplicate records removed: 261,831
- Final records: 738,332
- Missing values: 0
- ZIP prefixes associated with multiple states: 8 (flagged for review)
- Coordinate corrections: Not performed
- Latitude and longitude retained without modification.

The dataset was cleaned by removing exact duplicate rows.
Suspicious geographic relationships were preserved for traceability.